# Bayesian Change Point Detection for Brent Oil Prices
## Task 2: Change Point Modeling and Event Association

**Project:** Brent Oil Price Change Point Analysis  
**Organization:** Birhan Energies  
**Date:** February 8, 2026  
**Objective:** Detect structural breaks in oil prices and associate with geopolitical events

---

### Notebook Overview
1. Data Preparation
2. Bayesian Model Specification
3. MCMC Sampling
4. Convergence Diagnostics
5. Posterior Analysis
6. Event Association
7. Impact Quantification

## 1. Setup and Imports

In [1]:
# Standard libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Bayesian modeling
import pymc as pm
import arviz as az

# Custom modules
import sys
sys.path.insert(0, '..')
from src.models.bayesian_changepoint import BayesianChangePointModel

# Plotting configuration
plt.style.use('default')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

print('✓ Libraries loaded successfully')
print(f'PyMC version: {pm.__version__}')
print(f'ArviZ version: {az.__version__}')

✓ Libraries loaded successfully
PyMC version: 5.27.1
ArviZ version: 0.23.4


## 2. Load and Prepare Data

In [3]:
# Load processed data with features
df = pd.read_csv('../data/processed/prices_with_features.csv', 
                 index_col='Date', parse_dates=True)

print(f'Data shape: {df.shape}')
print(f'Date range: {df.index.min()} to {df.index.max()}')
print(f'\nColumns: {df.columns.tolist()}')

# Display sample
df.head()

Data shape: (9011, 9)
Date range: 1987-05-20 00:00:00 to 2022-11-14 00:00:00

Columns: ['Price', 'MA_30', 'MA_365', 'Rolling_Std', 'Log_Return', 'Volatility_30', 'Volatility_90', 'Month', 'Decade']


,Price,MA_30,MA_365,Rolling_Std,Log_Return,Volatility_30,Volatility_90,Month,Decade
Date,,,,,,,,,
1987-05-20,18.63,NaN,NaN,NaN,NaN,NaN,NaN,5,1980
1987-05-21,18.45,NaN,NaN,NaN,-0.009709,NaN,NaN,5,1980
1987-05-22,18.55,NaN,NaN,NaN,0.005405,NaN,NaN,5,1980
1987-05-25,18.60,NaN,NaN,NaN,0.002692,NaN,NaN,5,1980
1987-05-26,18.63,NaN,NaN,NaN,0.001612,NaN,NaN,5,1980


In [ ]:
# Use log returns (stationary series)
log_returns = df['log_return'].dropna()
dates = log_returns.index

print(f'Log returns: {len(log_returns)} observations')
print(f'\nStatistics:')
print(log_returns.describe())

# Plot log returns
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(dates, log_returns, linewidth=0.5, alpha=0.7, color='navy')
ax.axhline(y=0, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('Date', fontsize=12, fontweight='bold')
ax.set_ylabel('Log Returns', fontsize=12, fontweight='bold')
ax.set_title('Brent Oil Price Log Returns (Stationary Series)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Build Bayesian Change Point Model

In [ ]:
# Initialize model
model = BayesianChangePointModel(log_returns.values, dates=dates)

# Build model (mean shift)
pm_model = model.build_model(model_type='mean_shift')

# Display model structure
pm.model_to_graphviz(pm_model)

### Model Specification

**Priors:**
- τ (change point): DiscreteUniform(0, n_obs-1)
- μ_before: Normal(0, 0.1)
- μ_after: Normal(0, 0.1)
- σ: HalfNormal(0.1)

**Likelihood:**
- obs ~ Normal(μ(t), σ)
- where μ(t) = μ_before if t < τ, else μ_after

## 4. MCMC Sampling

In [ ]:
# Run MCMC sampling
# This may take 5-10 minutes depending on your machine
print('Starting MCMC sampling...')
print('This may take several minutes. Please wait...')

trace = model.sample(draws=2000, tune=1000, chains=4, target_accept=0.95)

print('\n✓ Sampling complete!')

## 5. Convergence Diagnostics

In [ ]:
# Summary statistics
summary = az.summary(trace, hdi_prob=0.95)
print('Posterior Summary:')
print(summary)

# Check convergence
print(f'\nConvergence Check:')
print(f'Max R-hat: {summary["r_hat"].max():.4f} (should be < 1.01)')
print(f'Min ESS: {summary["ess_bulk"].min():.0f} (should be > 400)')

In [ ]:
# Trace plots
az.plot_trace(trace, figsize=(14, 10))
plt.tight_layout()
plt.show()

## 6. Posterior Analysis

In [ ]:
# Get change point summary
cp_summary = model.get_change_point_summary()

print('Change Point Detection Results:')
print('='*60)
print(f'Most likely change point: {cp_summary["date_mode"]}')
print(f'Index: {cp_summary["tau_mode"]}')
print(f'95% HPD interval: [{cp_summary["date_hpd_lower"]}, {cp_summary["date_hpd_upper"]}]')
print(f'Posterior probability at mode: {cp_summary["probability_mass"]:.2%}')
print('='*60)

In [ ]:
# Plot posterior distribution of change point
model.plot_posterior()
plt.show()

In [ ]:
# Get parameter estimates
params = model.get_parameter_estimates()

print('Parameter Estimates:')
print('='*60)
for param, values in params.items():
    print(f'{param}:')
    print(f'  Mean: {values["mean"]:.6f}')
    print(f'  Std Dev: {values["sd"]:.6f}')
    print(f'  95% CI: [{values["hdi_lower"]:.6f}, {values["hdi_upper"]:.6f}]')
    print()
print('='*60)

In [ ]:
# Plot data with detected change point
model.plot_data_with_changepoint()
plt.show()

## 7. Event Association

In [ ]:
# Load events
events_df = pd.read_csv('../data/external/geopolitical_events.csv')
events_df['date'] = pd.to_datetime(events_df['date'])

print(f'Loaded {len(events_df)} events')
print(f'\nEvent categories:')
print(events_df['category'].value_counts())

events_df.head(10)

In [ ]:
# Associate change point with events (30-day window)
associated_events = model.associate_with_events(events_df, window_days=30)

if len(associated_events) > 0:
    print(f'Found {len(associated_events)} events within 30 days of change point:')
    print('='*80)
    for _, event in associated_events.iterrows():
        print(f'Date: {event["date"].date()}')
        print(f'Event: {event["event_name"]}')
        print(f'Category: {event["category"]}')
        print(f'Days from change point: {event["days_from_changepoint"]:+d}')
        print(f'Description: {event["description"]}')
        print('-'*80)
else:
    print('No events found within 30-day window')
    print('Try expanding the window or checking event dates')

## 8. Impact Quantification

In [ ]:
# Calculate impact metrics
if 'mu_before' in params and 'mu_after' in params:
    mu_before = params['mu_before']['mean']
    mu_after = params['mu_after']['mean']
    
    # Mean shift
    mean_shift = mu_after - mu_before
    mean_shift_pct = (mean_shift / abs(mu_before)) * 100 if mu_before != 0 else np.inf
    
    # Convert to price impact (approximate)
    # Log return change translates to percentage price change
    price_impact_pct = (np.exp(mean_shift) - 1) * 100
    
    print('Impact Quantification:')
    print('='*60)
    print(f'Mean log return before: {mu_before:.6f}')
    print(f'Mean log return after: {mu_after:.6f}')
    print(f'Change in mean: {mean_shift:+.6f}')
    print(f'Approximate price impact: {price_impact_pct:+.2f}%')
    print('='*60)
    
    # Probability statements
    mu_before_samples = trace.posterior['mu_before'].values.flatten()
    mu_after_samples = trace.posterior['mu_after'].values.flatten()
    
    prob_increase = (mu_after_samples > mu_before_samples).mean()
    
    print(f'\nProbabilistic Statement:')
    print(f'P(μ_after > μ_before | data) = {prob_increase:.2%}')
    
    if prob_increase > 0.95:
        print('Strong evidence of mean increase after change point')
    elif prob_increase < 0.05:
        print('Strong evidence of mean decrease after change point')
    else:
        print('Inconclusive evidence of mean change')

## 9. Posterior Predictive Checks

In [ ]:
# Posterior predictive check
with model.model:
    ppc = pm.sample_posterior_predictive(trace, random_seed=42)

# Plot
az.plot_ppc(az.from_pymc3(posterior_predictive=ppc, model=model.model), 
            num_pp_samples=100, figsize=(14, 6))
plt.tight_layout()
plt.show()

print('Posterior predictive check: Model should reproduce observed data distribution')

## 10. Summary and Conclusions

In [ ]:
print('BAYESIAN CHANGE POINT ANALYSIS - SUMMARY')
print('='*80)
print(f'\n1. CHANGE POINT DETECTION')
print(f'   Most likely date: {cp_summary["date_mode"]}')
print(f'   Confidence: {cp_summary["probability_mass"]:.1%} posterior probability')
print(f'   95% credible interval: [{cp_summary["date_hpd_lower"]}, {cp_summary["date_hpd_upper"]}]')

print(f'\n2. PARAMETER CHANGES')
if 'mu_before' in params and 'mu_after' in params:
    print(f'   Mean return before: {params["mu_before"]["mean"]:.6f}')
    print(f'   Mean return after: {params["mu_after"]["mean"]:.6f}')
    print(f'   Change: {mean_shift:+.6f} ({price_impact_pct:+.2f}% price impact)')

print(f'\n3. EVENT ASSOCIATION')
if len(associated_events) > 0:
    print(f'   {len(associated_events)} event(s) within 30-day window:')
    for _, event in associated_events.iterrows():
        print(f'   - {event["event_name"]} ({event["days_from_changepoint"]:+d} days)')
else:
    print(f'   No events within 30-day window')

print(f'\n4. MODEL DIAGNOSTICS')
print(f'   Convergence: R-hat = {summary["r_hat"].max():.4f} (✓ if < 1.01)')
print(f'   Sample size: ESS = {summary["ess_bulk"].min():.0f} (✓ if > 400)')

print(f'\n5. INTERPRETATION')
print(f'   The Bayesian model detected a structural break with high confidence.')
if len(associated_events) > 0:
    print(f'   This break coincides temporally with documented geopolitical/economic events.')
    print(f'   Temporal alignment suggests (but does not prove) causal relationship.')
print(f'   Additional evidence needed for definitive causal claims.')

print('\n' + '='*80)
print('ANALYSIS COMPLETE')
print('='*80)

## Next Steps

### Extensions to Consider:
1. **Multiple Change Points:** Extend model to detect multiple breaks
2. **Variance Shifts:** Model changes in volatility as well as mean
3. **Robust Likelihoods:** Use Student-t distribution for fat tails
4. **Multivariate Models:** Incorporate macroeconomic covariates
5. **Online Detection:** Real-time change point monitoring

### For Task 3 (Dashboard):
- Serve model results via Flask API
- Interactive visualization of posterior distributions
- Event filtering and exploration
- Scenario analysis tools